[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/080.%20The%20International%20Freight%20Mode%20Selection%20Problem%20%28Air%20vs.%20Ocean%29/P80-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 80. The International Freight Mode Selection Problem (Air vs. Ocean)
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Binary decision: each shipment uses either air or ocean freight
- Transportation costs are linear with weight
- Inventory holding costs apply for early arrivals
- Late delivery penalties are linear with delay days
- Air freight has limited capacity constraints
- Market conditions are deterministic for planning horizon

### Approach (step-by-step)
1. **Define decision variables**: Binary variables for each shipment-mode-destination-period combination
2. **Formulate objective function**: Minimize total transportation + holding + penalty costs
3. **Add constraints**: Assignment, capacity, and time relationship constraints
4. **Solve using mixed-integer programming**: Use pulp solver for exact optimization
5. **Analyze results**: Extract optimal mode assignments and cost breakdown

### What to look for in the results
- Optimal mode assignments balancing cost and time sensitivity
- Cost breakdown between transportation, holding, and penalties
- Capacity utilization for air freight
- Trade-offs between cost efficiency and service levels

### Concrete example (from the source)
TechGlobal Manufacturing must optimize freight mode selection for three shipments:
- Shipment 1: 500kg microprocessors, value $2M, due in Rotterdam in 14 days
- Shipment 2: 2000kg components, value $500K, due in LA in 21 days  
- Shipment 3: 5000kg servers, value $1M, due in Rotterdam in 35 days

Transportation costs:
- Ocean: $2/kg to Rotterdam, $1.8/kg to LA
- Air: $6.2/kg to Rotterdam, $5.5/kg to LA

Holding cost: 0.1%/day of shipment value
Late penalty: $1000/day
Air capacity limit: 8000kg/month

In [ ]:
# Import required packages for mathematical optimization
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Shipment:
    """Data class for shipment information"""
    id: int
    weight: float  # kg
    value: float   # USD
    destination: str
    due_date: int  # days from now

@dataclass 
class TransportParams:
    """Parameters for transportation costs and times"""
    costs: Dict[str, Dict[str, float]]  # mode -> destination -> cost per kg
    transit_times: Dict[str, Dict[str, int]]  # mode -> destination -> days
    holding_rate: float  # daily holding cost rate as fraction of value
    late_penalty: float  # penalty per day late
    air_capacity: float  # monthly air freight capacity in kg

# Define the concrete example from the problem description
shipments = [
    Shipment(1, 500, 2000000, "rotterdam", 14),   # High-value microprocessors
    Shipment(2, 2000, 500000, "los_angeles", 21),  # Standard components
    Shipment(3, 5000, 1000000, "rotterdam", 35),   # Large servers
]

params = TransportParams(
    costs={
        "ocean": {"rotterdam": 2.0, "los_angeles": 1.8},
        "air": {"rotterdam": 6.2, "los_angeles": 5.5}
    },
    transit_times={
        "ocean": {"rotterdam": 25, "los_angeles": 18},
        "air": {"rotterdam": 3, "los_angeles": 2}
    },
    holding_rate=0.001,  # 0.1% per day
    late_penalty=1000,   # $1000 per day
    air_capacity=8000   # 8000 kg per month
)

print("Shipments:")
for s in shipments:
    print(f"  Shipment {s.id}: {s.weight}kg, ${s.value:,}, to {s.destination}, due in {s.due_date} days")
print(f"\nAir freight capacity: {params.air_capacity:,} kg/month")

In [ ]:
def calculate_total_cost(shipment: Shipment, mode: str, params: TransportParams) -> float:
    """
    Calculate total cost for a single shipment using a specific transport mode
    Includes transportation cost, holding cost (if early), and late penalty (if late)
    """
    # Transportation cost
    transport_cost = shipment.weight * params.costs[mode][shipment.destination]
    
    # Transit time
    transit_time = params.transit_times[mode][shipment.destination]
    
    # Holding cost if arrives early
    if transit_time < shipment.due_date:
        early_days = shipment.due_date - transit_time
        holding_cost = early_days * params.holding_rate * shipment.value
    else:
        holding_cost = 0
    
    # Late penalty if arrives after due date
    if transit_time > shipment.due_date:
        late_days = transit_time - shipment.due_date
        late_penalty_cost = late_days * params.late_penalty
    else:
        late_penalty_cost = 0
    
    return transport_cost + holding_cost + late_penalty_cost

# Test cost calculation for all mode-shipment combinations
print("Cost Analysis:")
print("=" * 80)
for shipment in shipments:
    print(f"\nShipment {shipment.id} ({shipment.destination}):")
    for mode in ['ocean', 'air']:
        cost = calculate_total_cost(shipment, mode, params)
        transit = params.transit_times[mode][shipment.destination]
        print(f"  {mode.capitalize():6}: ${cost:,.0f} (transit: {transit} days)")

In [ ]:
def solve_freight_mode_mip(shipments: List[Shipment], params: TransportParams) -> Tuple[pulp.LpProblem, Dict]:
    """
    Solve the freight mode selection problem using Mixed-Integer Programming
    Returns the optimization problem and solution results
    """
    # Create the optimization problem
    prob = pulp.LpProblem("Freight_Mode_Selection", pulp.LpMinimize)
    
    # Decision variables: x[shipment_id][mode] = 1 if shipment uses mode, 0 otherwise
    x = {}
    for shipment in shipments:
        for mode in ['ocean', 'air']:
            var_name = f"x_{shipment.id}_{mode}"
            x[(shipment.id, mode)] = pulp.LpVariable(var_name, cat='Binary')
    
    # Early and late arrival variables (for linearization)
    early = {}
    late = {}
    for shipment in shipments:
        early[shipment.id] = pulp.LpVariable(f"early_{shipment.id}", lowBound=0)
        late[shipment.id] = pulp.LpVariable(f"late_{shipment.id}", lowBound=0)
    
    # Objective function: minimize total cost
    total_cost = 0
    
    # Transportation costs
    for shipment in shipments:
        for mode in ['ocean', 'air']:
            transport_cost = shipment.weight * params.costs[mode][shipment.destination]
            total_cost += transport_cost * x[(shipment.id, mode)]
    
    # Holding costs and late penalties
    for shipment in shipments:
        holding_cost = params.holding_rate * shipment.value * early[shipment.id]
        penalty_cost = params.late_penalty * late[shipment.id]
        total_cost += holding_cost + penalty_cost
    
    prob += total_cost, "Total_Supply_Chain_Cost"
    
    # Constraint 1: Each shipment must use exactly one mode
    for shipment in shipments:
        prob += (x[(shipment.id, 'ocean')] + x[(shipment.id, 'air')] == 1, 
                f"assign_mode_{shipment.id}")
    
    # Constraint 2: Air freight capacity constraint
    air_weight = 0
    for shipment in shipments:
        air_weight += shipment.weight * x[(shipment.id, 'air')]
    prob += (air_weight <= params.air_capacity, "air_capacity")
    
    # Constraint 3: Time relationship constraints
    for shipment in shipments:
        for mode in ['ocean', 'air']:
            transit_time = params.transit_times[mode][shipment.destination]
            prob += (early[shipment.id] - late[shipment.id] == 
                    shipment.due_date - transit_time * x[(shipment.id, mode)], 
                    f"time_relationship_{shipment.id}_{mode}")
    
    # Solve the problem
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    return prob, x

# Solve the optimization problem
print("Solving Mixed-Integer Programming Problem...")
prob, decision_vars = solve_freight_mode_mip(shipments, params)

print(f"\nOptimization Status: {pulp.LpStatus[prob.status]}")
print(f"Optimal Total Cost: ${pulp.value(prob.objective):,.0f}")

In [ ]:
def extract_solution(shipments: List[Shipment], decision_vars: Dict, params: TransportParams) -> Dict:
    """
    Extract and format the solution results
    """
    results = {
        'assignments': {},
        'cost_breakdown': [],
        'total_cost': 0,
        'air_weight_used': 0
    }
    
    for shipment in shipments:
        # Get assigned mode
        mode = 'air' if pulp.value(decision_vars[(shipment.id, 'air')]) == 1 else 'ocean'
        results['assignments'][shipment.id] = mode
        
        if mode == 'air':
            results['air_weight_used'] += shipment.weight
        
        # Calculate cost components
        transport_cost = shipment.weight * params.costs[mode][shipment.destination]
        transit_time = params.transit_times[mode][shipment.destination]
        
        if transit_time < shipment.due_date:
            early_days = shipment.due_date - transit_time
            holding_cost = early_days * params.holding_rate * shipment.value
            late_cost = 0
        else:
            holding_cost = 0
            late_days = transit_time - shipment.due_date
            late_cost = late_days * params.late_penalty
        
        total_shipment_cost = transport_cost + holding_cost + late_cost
        
        results['cost_breakdown'].append({
            'shipment_id': shipment.id,
            'mode': mode,
            'transport_cost': transport_cost,
            'holding_cost': holding_cost,
            'late_cost': late_cost,
            'total_cost': total_shipment_cost,
            'transit_time': transit_time,
            'due_date': shipment.due_date
        })
        
        results['total_cost'] += total_shipment_cost
    
    return results

# Extract and display solution
results = extract_solution(shipments, decision_vars, params)

print("\nOptimal Solution:")
print("=" * 80)
for detail in results['cost_breakdown']:
    print(f"\nShipment {detail['shipment_id']}: {detail['mode'].upper()}")
    print(f"  Transport cost: ${detail['transport_cost']:,.0f}")
    print(f"  Holding cost: ${detail['holding_cost']:,.0f}")
    print(f"  Late penalty: ${detail['late_cost']:,.0f}")
    print(f"  Total cost: ${detail['total_cost']:,.0f}")
    print(f"  Transit time: {detail['transit_time']} days (due: {detail['due_date']})")

print(f"\nSummary:")
print(f"  Total cost: ${results['total_cost']:,.0f}")
print(f"  Air capacity used: {results['air_weight_used']:,} kg / {params.air_capacity:,} kg")
print(f"  Capacity utilization: {results['air_weight_used']/params.air_capacity*100:.1f}%")

In [ ]:
# Compare with baseline strategies
def compare_baseline_strategies(shipments: List[Shipment], params: TransportParams) -> Dict:
    """
    Compare optimal solution with simple baseline strategies
    """
    strategies = {}
    
    # Strategy 1: All ocean freight
    ocean_cost = sum(calculate_total_cost(s, 'ocean', params) for s in shipments)
    ocean_weight = sum(s.weight for s in shipments)
    strategies['all_ocean'] = {
        'cost': ocean_cost,
        'air_weight': 0,
        'description': 'All shipments via ocean freight'
    }
    
    # Strategy 2: All air freight (check capacity)
    total_air_weight = sum(s.weight for s in shipments)
    if total_air_weight <= params.air_capacity:
        air_cost = sum(calculate_total_cost(s, 'air', params) for s in shipments)
        strategies['all_air'] = {
            'cost': air_cost,
            'air_weight': total_air_weight,
            'description': 'All shipments via air freight'
        }
    else:
        strategies['all_air'] = {
            'cost': float('inf'),
            'air_weight': total_air_weight,
            'description': f'Infeasible: exceeds air capacity ({total_air_weight:,} > {params.air_capacity:,} kg)'
        }
    
    # Strategy 3: Optimal solution
    strategies['optimal'] = {
        'cost': results['total_cost'],
        'air_weight': results['air_weight_used'],
        'description': 'Mixed-integer programming optimal solution'
    }
    
    return strategies

# Compare strategies
strategies = compare_baseline_strategies(shipments, params)

print("\nStrategy Comparison:")
print("=" * 80)
for name, strategy in strategies.items():
    if strategy['cost'] == float('inf'):
        print(f"{name.replace('_', ' ').title()}: {strategy['description']}")
    else:
        print(f"{name.replace('_', ' ').title()}: ${strategy['cost']:,.0f}")
        print(f"  Air weight: {strategy['air_weight']:,} kg")
        print(f"  {strategy['description']}")
    print()

# Calculate savings
if strategies['all_ocean']['cost'] != float('inf'):
    ocean_savings = strategies['all_ocean']['cost'] - strategies['optimal']['cost']
    ocean_savings_pct = ocean_savings / strategies['all_ocean']['cost'] * 100
    print(f"Optimal vs All-Ocean: Save ${ocean_savings:,.0f} ({ocean_savings_pct:.1f}%)")

if strategies['all_air']['cost'] != float('inf'):
    air_savings = strategies['all_air']['cost'] - strategies['optimal']['cost']
    air_savings_pct = air_savings / strategies['all_air']['cost'] * 100 if strategies['all_air']['cost'] > 0 else 0
    print(f"Optimal vs All-Air: Save ${air_savings:,.0f} ({air_savings_pct:.1f}%)")

In [ ]:
# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Freight Mode Selection Optimization Analysis', fontsize=16, fontweight='bold')

# 1. Cost breakdown by shipment and mode
shipment_ids = [detail['shipment_id'] for detail in results['cost_breakdown']]
transport_costs = [detail['transport_cost'] for detail in results['cost_breakdown']]
holding_costs = [detail['holding_cost'] for detail in results['cost_breakdown']]
late_costs = [detail['late_cost'] for detail in results['cost_breakdown']]
modes = [detail['mode'].upper() for detail in results['cost_breakdown']]
colors = ['lightcoral' if mode == 'AIR' else 'lightblue' for mode in modes]

x = np.arange(len(shipment_ids))
width = 0.25

ax1.bar(x - width, transport_costs, width, label='Transport', alpha=0.8, color='steelblue')
ax1.bar(x, holding_costs, width, label='Holding', alpha=0.8, color='gold')
ax1.bar(x + width, late_costs, width, label='Late Penalty', alpha=0.8, color='crimson')

ax1.set_xlabel('Shipment ID')
ax1.set_ylabel('Cost ($)')
ax1.set_title('Cost Breakdown by Shipment')
ax1.set_xticks(x)
ax1.set_xticklabels([f'S{id}\n({mode})' for id, mode in zip(shipment_ids, modes)])
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Strategy comparison
strategy_names = []
strategy_costs = []
strategy_colors = []

for name, strategy in strategies.items():
    if strategy['cost'] != float('inf'):
        strategy_names.append(name.replace('_', '\n').title())
        strategy_costs.append(strategy['cost'])
        strategy_colors.append('green' if name == 'optimal' else 'orange')

bars = ax2.bar(strategy_names, strategy_costs, color=strategy_colors, alpha=0.7)
ax2.set_ylabel('Total Cost ($)')
ax2.set_title('Strategy Comparison')
ax2.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, cost in zip(bars, strategy_costs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')

ax2.grid(True, alpha=0.3)

# 3. Mode utilization
mode_counts = {'Air': 0, 'Ocean': 0}
mode_weights = {'Air': 0, 'Ocean': 0}

for detail in results['cost_breakdown']:
    mode = detail['mode'].title()
    mode_counts[mode] += 1
    shipment = next(s for s in shipments if s.id == detail['shipment_id'])
    mode_weights[mode] += shipment.weight

# Create pie chart for shipment count
ax3.pie(mode_counts.values(), labels=mode_counts.keys(), autopct='%1.0f shipments', 
        colors=['lightcoral', 'lightblue'], startangle=90)
ax3.set_title('Shipment Count by Mode')

# 4. Weight distribution and capacity
total_weight = sum(s.weight for s in shipments)
air_weight = mode_weights['Air']
ocean_weight = mode_weights['Ocean']

categories = ['Air Freight', 'Ocean Freight']
weights = [air_weight, ocean_weight]
colors = ['lightcoral', 'lightblue']

bars = ax4.bar(categories, weights, color=colors, alpha=0.7)
ax4.set_ylabel('Weight (kg)')
ax4.set_title('Weight Distribution by Transport Mode')
ax4.grid(True, alpha=0.3)

# Add capacity line for air freight
ax4.axhline(y=params.air_capacity, color='red', linestyle='--', 
            label=f'Air Capacity ({params.air_capacity:,} kg)', linewidth=2)
ax4.legend()

# Add value labels on bars
for bar, weight in zip(bars, weights):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'{weight:,.0f} kg', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nVisualization Summary:")
print("=" * 80)
print(f"1. Cost Breakdown: Shows transport, holding, and penalty costs for each shipment")
print(f"2. Strategy Comparison: Optimal solution vs baseline strategies")
print(f"3. Mode Utilization: {mode_counts['Air']} shipments by air, {mode_counts['Ocean']} by ocean")
print(f"4. Weight Distribution: {air_weight:,} kg air freight ({air_weight/params.air_capacity*100:.1f}% capacity)")

In [ ]:
# Sensitivity analysis on key parameters
def sensitivity_analysis(shipments: List[Shipment], params: TransportParams) -> Dict:
    """
    Perform sensitivity analysis on key parameters
    """
    sensitivity_results = {}
    
    # Test different air capacity levels
    capacity_levels = [4000, 6000, 8000, 10000, 12000]
    capacity_costs = []
    
    for capacity in capacity_levels:
        test_params = TransportParams(
            costs=params.costs,
            transit_times=params.transit_times,
            holding_rate=params.holding_rate,
            late_penalty=params.late_penalty,
            air_capacity=capacity
        )
        _, test_vars = solve_freight_mode_mip(shipments, test_params)
        test_results = extract_solution(shipments, test_vars, test_params)
        capacity_costs.append(test_results['total_cost'])
    
    sensitivity_results['capacity'] = {
        'levels': capacity_levels,
        'costs': capacity_costs
    }
    
    # Test different holding cost rates
    holding_rates = [0.0005, 0.001, 0.0015, 0.002, 0.0025]  # 0.05% to 0.25%
    holding_costs = []
    
    for rate in holding_rates:
        test_params = TransportParams(
            costs=params.costs,
            transit_times=params.transit_times,
            holding_rate=rate,
            late_penalty=params.late_penalty,
            air_capacity=params.air_capacity
        )
        _, test_vars = solve_freight_mode_mip(shipments, test_params)
        test_results = extract_solution(shipments, test_vars, test_params)
        holding_costs.append(test_results['total_cost'])
    
    sensitivity_results['holding_rate'] = {
        'levels': holding_rates,
        'costs': holding_costs
    }
    
    return sensitivity_results

# Perform sensitivity analysis
print("Performing Sensitivity Analysis...")
sensitivity = sensitivity_analysis(shipments, params)

# Create sensitivity plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sensitivity Analysis', fontsize=16, fontweight='bold')

# Air capacity sensitivity
ax1.plot(sensitivity['capacity']['levels'], sensitivity['capacity']['costs'], 
         'o-', linewidth=2, markersize=8, color='steelblue')
ax1.set_xlabel('Air Freight Capacity (kg)')
ax1.set_ylabel('Total Cost ($)')
ax1.set_title('Impact of Air Capacity on Total Cost')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=params.air_capacity, color='red', linestyle='--', 
            label=f'Current ({params.air_capacity:,} kg)', linewidth=2)
ax1.legend()

# Holding rate sensitivity
holding_pct = [rate * 100 for rate in sensitivity['holding_rate']['levels']]
ax2.plot(holding_pct, sensitivity['holding_rate']['costs'], 
         'o-', linewidth=2, markersize=8, color='darkorange')
ax2.set_xlabel('Holding Cost Rate (% per day)')
ax2.set_ylabel('Total Cost ($)')
ax2.set_title('Impact of Holding Cost Rate on Total Cost')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=params.holding_rate * 100, color='red', linestyle='--', 
            label=f'Current ({params.holding_rate*100:.2f}%)', linewidth=2)
ax2.legend()

plt.tight_layout()
plt.show()

print("\nSensitivity Analysis Results:")
print("=" * 80)
print(f"Air Capacity Impact:")
for i, (capacity, cost) in enumerate(zip(sensitivity['capacity']['levels'], sensitivity['capacity']['costs'])):
    change = cost - sensitivity['capacity']['costs'][2]  # Compare to baseline (8000 kg)
    print(f"  {capacity:,} kg: ${cost:,.0f} ({change:+,.0f})")

print(f"\nHolding Rate Impact:")
for i, (rate, cost) in enumerate(zip(sensitivity['holding_rate']['levels'], sensitivity['holding_rate']['costs'])):
    change = cost - sensitivity['holding_rate']['costs'][1]  # Compare to baseline (0.1%)
    print(f"  {rate*100:.2f}%: ${cost:,.0f} ({change:+,.0f})")

### Why this Tier exists vs earlier Tiers
This is the foundational Tier that establishes the mathematical framework for freight mode selection. It provides:
- **Exact optimality guarantees** through mixed-integer programming
- **Comprehensive cost modeling** including transport, holding, and penalty components
- **Constraint handling** for capacity and service requirements
- **Benchmark for comparison** with heuristic and advanced methods

### Pros / Cons vs this Tier
**Pros:**
- Guarantees optimal solution for given problem instance
- Handles complex constraints and multiple objectives
- Provides sensitivity analysis capabilities
- Transparent and explainable decision logic

**Cons:**
- Computationally intensive for large problem instances
- Requires deterministic parameters
- Limited scalability for real-time decisions
- Complex formulation for non-technical users

### When to use this Tier
- **Strategic planning** where optimality is critical
- **Medium-scale problems** (up to ~100 shipments)
- **Regulatory compliance** requiring documented optimization
- **Benchmark development** for evaluating heuristic methods
- **Contract negotiations** requiring cost justification